In [5]:
from glob import iglob
import sys
from loguru import logger

note_id_to_person_csvs_path = (
    "/home/westerd/_/project_data/tedla-hypertension/source_data/note_id_to_person"
)

logger.remove()
logger.add(
    "/home/westerd/_/project_data/tedla-hypertension/logs/note_id_to_person.log",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}",
    enqueue=True,
)

logger.info(f"Processing CSV files from: {note_id_to_person_csvs_path}")

In [6]:
import os
import duckdb
import sys

sys.path.append(
    "/home/westerd/_/research_projects/tedla-hypertension/src/nlp_method/notebooks"
)

results_db_path = "/home/westerd/_/project_data/tedla-hypertension/results/results.db"
cohort_note_data_table = os.getenv("COHORT_NOTE_DATA_TABLE", "results")
assert cohort_note_data_table is not None

duckdb_cxn = duckdb.connect()
duckdb_cxn.install_extension("sqlite")
duckdb_cxn.load_extension("sqlite")
duckdb_cxn.execute(f"ATTACH '{results_db_path}' as results_db (TYPE sqlite);")
duckdb_cxn.sql("SHOW TABLES FROM results_db;").df().head(10)

# with sqlite3.connect(results_db_path) as sqlite_cxn:

duckdb_cxn.sql(f"PRAGMA table_info(results_db.{cohort_note_data_table});").df()

,cid,name,type,notnull,dflt_value,pk
0,0,result_id,BIGINT,False,None,True
1,1,note_id,BIGINT,False,None,False
2,2,batch_group,BIGINT,False,None,False
3,3,window_text,VARCHAR,False,None,False
4,4,search_term_to_note_groups_id,BIGINT,False,None,False
5,5,search_term,VARCHAR,False,None,False
6,6,window_start_char_offset,BIGINT,False,None,False
7,7,window_end_char_offset,BIGINT,False,None,False
8,8,entity_start_offset,BIGINT,False,None,False
9,9,entity_end_offset,BIGINT,False,None,False


In [7]:
from note_id_to_person_proc import persist_note_id_mmap

note_ids_mmap = "/home/westerd/_/project_data/tedla-hypertension/results/note_ids.bin"
persist_note_id_mmap(note_ids_mmap, results_db_path, cohort_note_data_table)

In [ ]:
from glob import glob
from itertools import repeat
from typing import List
from note_id_to_person_proc import process_note_id_to_person_csv
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from tqdm import tqdm
import pandas as pd
from concurrent.futures import as_completed


def create_csv_iter(
    note_id_to_person_csvs_path, max_files: int | None = None
) -> List[str]:
    it = glob(f"{note_id_to_person_csvs_path}/part-*.csv")
    if max_files is not None:
        return [f for _, f in zip(range(max_files), it)]
    return it


def count_csv_files(note_id_to_person_csvs_path):
    return sum(1 for _ in create_csv_iter(note_id_to_person_csvs_path))


MAX_FILES_TO_TEST = None

files_to_process = create_csv_iter(
    note_id_to_person_csvs_path, max_files=MAX_FILES_TO_TEST
)

max_workers = min(100, MAX_FILES_TO_TEST or 100)
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    list(
        tqdm(
            executor.map(
                process_note_id_to_person_csv, files_to_process, repeat(note_ids_mmap)
            ),
            total=len(files_to_process),
            desc="Processing CSV files",
        )
    )
    
logger.success("All CSV files have been processed.")

Processing CSV files: 100%|██████████| 200/200 [00:28<00:00,  7.14it/s]


<loguru._logger.Logger.complete.<locals>.AwaitableCompleter at 0x7f44b2ac3890>

In [ ]:
import pandas as pd
from glob import glob

parquet_dir = "/home/westerd/_/project_data/tedla-hypertension/note_id_to_person_output"
parquet_files = glob(f"{parquet_dir}/*.parquet")

df_all = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
final_parquet = f"{parquet_dir}/final.parquet"
df_all.to_parquet(final_parquet)

In [11]:
import os
from glob import glob

filtered_files = glob("/home/westerd/_/project_data/tedla-hypertension/note_id_to_person_output/*.filtered.parquet")
for f in filtered_files:
    os.remove(f)
logger.info(f"Removed {len(filtered_files)} filtered parquet files.")

In [ ]:
pd.read_parquet(final_parquet).head(10)

,person_id,deid_pat_id,mrn,note_id
0,5079024,971113,17446196,-838312277324451469
1,5079762,629600,32149643,-6278386640111695929
2,5079762,629600,32149643,8460026642716707853
3,5079762,629600,32149643,475317877811646292
4,5080232,1111253,41415878,2893100303500955765
5,5080232,1111253,41415878,-4956486363994176895
6,5080232,1111253,41415878,6688788272206888557
7,5080232,1111253,41415878,109213401274900778
8,5086441,518478,38568317,5894200275387407949
9,5096468,193357,14756936,1488215764053280828


In [14]:
unique_person_ids = df_all['person_id'].nunique()
print(f"Total unique person_id values: {unique_person_ids}")

Total unique person_id values: 671497


In [15]:

df_all.sort_values(by=["person_id", "note_id"]).to_csv("/home/westerd/_/project_data/tedla-hypertension/note_id_to_person_output/note_id_to_person.csv", index=False)